# INCA vs KENDA-CH1 altitude comparison

Maps INCA forecast elevation onto the KENDA-CH1 truth grid using the same nearest-neighbour
approach as the evalml verification pipeline, then plots both elevation fields and their
difference — highlighting grid boxes where |Δz| > 1000 m.
The black rectangle marks the `'all'` inner verification domain.

In [ ]:
import os
import sys

# Must be set before any earthkit/eccodes import
os.environ["ECCODES_DEFINITION_PATH"] = os.path.realpath(
    ".venv/share/eccodes-cosmo-resources/definitions"
)
sys.path.insert(0, "src")

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from data_input import load_forecast_data, load_truth_data
from verification.spatial import map_forecast_to_truth

In [ ]:
REFTIME = datetime(2025, 3, 1)
STEPS   = [0, 6]          # elevation is time-invariant; step 0 is sufficient
PARAMS  = ["T_2M"]    # minimal param; we only need the elevation coordinate

INCA_ROOT  = Path("/store_new/mch/msclim/INCA")
KENDA_ROOT = Path("/store_new/mch/msopr/ml/datasets/mch-ich1-1km-2024-2025-1h-pl13-v1.0.zarr")

In [ ]:
print("Loading INCA forecast...")
fcst = load_forecast_data(INCA_ROOT, REFTIME, STEPS, PARAMS)
print(fcst)

print("\nLoading KENDA-CH1 truth...")
truth = load_truth_data(KENDA_ROOT, REFTIME, STEPS, PARAMS)
print(truth)

In [ ]:
print("Mapping INCA onto KENDA-CH1 grid (nearest neighbour)...")
fcst_mapped = map_forecast_to_truth(fcst, truth)
print(fcst_mapped)

In [ ]:
# Extract elevation arrays — same convention as apply_lapse_rate_correction:
#   dz = obs["elevation"] - fcst["elevation"]
kenda_elev = truth["elevation"].values        # KENDA-CH1 elevation at truth grid points
inca_elev  = fcst_mapped["elevation"].values  # INCA elevation at nearest KENDA point
lat = truth["latitude"].values
lon = truth["longitude"].values

dz = kenda_elev - inca_elev

print(f"KENDA-CH1 elevation:    min={kenda_elev.min():.0f} m, max={kenda_elev.max():.0f} m")
print(f"INCA elevation (mapped): min={inca_elev.min():.0f} m, max={inca_elev.max():.0f} m")
print(f"Δz (KENDA − INCA):      range [{dz.min():.1f}, {dz.max():.1f}] m, mean {dz.mean():.1f} m")
n_large = int((np.abs(dz) > 1000).sum())
pct_large = 100 * n_large / len(dz)
print(f"Points with |Δz| > 1000 m: {n_large} ({pct_large:.1f}%)")

In [ ]:
PROJ   = ccrs.PlateCarree()
extent = [lon.min() - 0.2, lon.max() + 0.2, lat.min() - 0.2, lat.max() + 0.2]

# 'all' inner verification domain (lon [1.5, 16], lat [43, 49.5])
# defined in ShapefileSpatialAggregationMasks in src/verification/__init__.py
ALL_LON = (1.5, 16.0)
ALL_LAT = (43.0, 49.5)

def _add_all_region(ax):
    rect = mpatches.Rectangle(
        xy=(ALL_LON[0], ALL_LAT[0]),
        width=ALL_LON[1] - ALL_LON[0],
        height=ALL_LAT[1] - ALL_LAT[0],
        linewidth=1.8, edgecolor="black", facecolor="none",
        transform=PROJ, zorder=4, label="'all' verification domain",
    )
    ax.add_patch(rect)

def _decorate(ax, legend=False):
    ax.set_extent(extent, crs=PROJ)
    ax.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor="0.3")
    ax.add_feature(cfeature.LAKES, facecolor="lightblue", edgecolor="none", alpha=0.6)
    ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, x_inline=False, y_inline=False)
    _add_all_region(ax)
    if legend:
        ax.legend(loc="lower right", fontsize=8, markerscale=8, framealpha=0.8)

fig, axes = plt.subplots(
    1, 3, figsize=(18, 5),
    subplot_kw={"projection": PROJ},
)

scatter_kw = dict(s=0.05, transform=PROJ, rasterized=True)

# --- Panel 1: KENDA-CH1 elevation ---
sc1 = axes[0].scatter(lon, lat, c=kenda_elev, cmap="terrain", vmin=0, vmax=3500, **scatter_kw)
plt.colorbar(sc1, ax=axes[0], label="Elevation (m)", shrink=0.85, pad=0.02)
axes[0].set_title("KENDA-CH1 elevation")
_decorate(axes[0])

# --- Panel 2: INCA elevation (mapped to KENDA grid) ---
sc2 = axes[1].scatter(lon, lat, c=inca_elev, cmap="terrain", vmin=0, vmax=3500, **scatter_kw)
plt.colorbar(sc2, ax=axes[1], label="Elevation (m)", shrink=0.85, pad=0.02)
axes[1].set_title("INCA elevation (nearest-neighbour mapped)")
_decorate(axes[1])

# --- Panel 3: Δz = KENDA − INCA, |Δz| > 1000 m highlighted in yellow ---
sc3 = axes[2].scatter(lon, lat, c=dz, cmap="RdBu_r", vmin=-2000, vmax=2000, **scatter_kw)
plt.colorbar(sc3, ax=axes[2], label="Δz = KENDA − INCA (m)", shrink=0.85, pad=0.02)
mask = np.abs(dz) > 1000
axes[2].scatter(
    lon[mask], lat[mask], c="yellow", s=0.3, transform=PROJ,
    label=f"|Δz| > 1000 m (n={mask.sum():,})", zorder=3, rasterized=True,
)
axes[2].set_title("Elevation difference (KENDA − INCA)")
_decorate(axes[2], legend=True)

fig.suptitle(
    f"INCA vs KENDA-CH1 elevation — INCA nearest-neighbour mapped to KENDA-CH1 grid\n"
    f"reftime {REFTIME:%Y-%m-%d}  |  Δz = obs − fcst  |  black box: 'all' verification domain",
    fontsize=11,
)
plt.tight_layout()
plt.savefig("output/altitude_comparison_inca_kenda.png", dpi=150, bbox_inches="tight")
plt.show()

## INCA domain extent and largest inscribed lat/lon box

The INCA grid is regular in projected space (LV95) but its edges form slightly curved lines
in lat/lon. The algorithm below finds the largest axis-aligned lat/lon rectangle fully
enclosed by the INCA domain by scanning all row pairs and tracking the tightest lon bounds.

In [ ]:
# INCA lat/lon are 2-D coordinates on the (y, x) grid
lat2d = fcst["latitude"].values   # (ny, nx)
lon2d = fcst["longitude"].values  # (ny, nx)

# Per-row (constant y) bounds in lat and lon
row_min_lon = lon2d.min(axis=1)
row_max_lon = lon2d.max(axis=1)
row_mean_lat = lat2d.mean(axis=1)

# Sort rows by increasing mean latitude so the range search is monotone
order = np.argsort(row_mean_lat)
s_lat     = row_mean_lat[order]
s_min_lon = row_min_lon[order]
s_max_lon = row_max_lon[order]

ny = len(s_lat)
step = max(1, ny // 128)   # sample ~128 rows for an O(128²) search
idx  = np.arange(0, ny, step)

best_area = 0.0
best_box  = None   # (lon_lo, lon_hi, lat_lo, lat_hi)

for ii, i in enumerate(idx):
    cur_min_lon = s_min_lon[i]
    cur_max_lon = s_max_lon[i]
    lat_lo = s_lat[i]
    for j in idx[ii + 1:]:
        # Tighten lon bounds to stay within every row in [i, j]
        cur_min_lon = max(cur_min_lon, s_min_lon[i:j+1].max())
        cur_max_lon = min(cur_max_lon, s_max_lon[i:j+1].min())
        if cur_max_lon <= cur_min_lon:
            break   # domain has narrowed to nothing — no wider box possible
        lat_hi = s_lat[j]
        area = (lat_hi - lat_lo) * (cur_max_lon - cur_min_lon)
        if area > best_area:
            best_area = area
            best_box  = (cur_min_lon, cur_max_lon, lat_lo, lat_hi)

lon_lo, lon_hi, lat_lo, lat_hi = best_box
print("Largest inscribed lat/lon box fully within the INCA domain:")
print(f"  lon: [{lon_lo:.3f}, {lon_hi:.3f}]")
print(f"  lat: [{lat_lo:.3f}, {lat_hi:.3f}]")
print(f"  area: {best_area:.3f} deg²")

In [ ]:
# Boundary of the INCA domain: first/last rows and columns
boundary_lat = np.concatenate([
    lat2d[0, :],   lat2d[:, -1],
    lat2d[-1, ::-1], lat2d[::-1, 0],
])
boundary_lon = np.concatenate([
    lon2d[0, :],   lon2d[:, -1],
    lon2d[-1, ::-1], lon2d[::-1, 0],
])

inca_extent = [
    lon2d.min() - 0.5, lon2d.max() + 0.5,
    lat2d.min() - 0.5, lat2d.max() + 0.5,
]

fig, ax = plt.subplots(figsize=(9, 7), subplot_kw={"projection": PROJ})
ax.set_extent(inca_extent, crs=PROJ)
ax.add_feature(cfeature.BORDERS, linewidth=0.7, edgecolor="0.4")
ax.add_feature(cfeature.LAKES,   facecolor="lightblue", edgecolor="none", alpha=0.5)
ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5, x_inline=False, y_inline=False)

# INCA domain fill (light grey) and boundary (dark)
ax.fill(
    boundary_lon, boundary_lat, transform=PROJ,
    facecolor="steelblue", alpha=0.25, label="INCA domain",
)
ax.plot(
    np.append(boundary_lon, boundary_lon[0]),
    np.append(boundary_lat, boundary_lat[0]),
    transform=PROJ, color="steelblue", linewidth=1.2,
)

# Proposed largest inscribed box (red)
inscribed = mpatches.Rectangle(
    xy=(lon_lo, lat_lo), width=lon_hi - lon_lo, height=lat_hi - lat_lo,
    linewidth=2.0, edgecolor="crimson", facecolor="none",
    transform=PROJ, zorder=4,
    label=(
        f"Largest inscribed box\n"
        f"lon [{lon_lo:.2f}, {lon_hi:.2f}]\n"
        f"lat [{lat_lo:.2f}, {lat_hi:.2f}]"
    ),
)
ax.add_patch(inscribed)

# 'all' verification domain (black)
_add_all_region(ax)

ax.legend(loc="lower right", fontsize=9, framealpha=0.85)
ax.set_title(
    "INCA domain extent, 'all' verification domain (black),\n"
    "and largest inscribed lat/lon box (red)",
    fontsize=11,
)
plt.tight_layout()
plt.savefig("output/inca_domain_inscribed_box.png", dpi=150, bbox_inches="tight")
plt.show()